# Paper PoRT Recreated Artifact Smoke Matrix

This notebook validates a `PORT_ARTIFACT_MODE=recreated` path on a small WMDP matrix before any full PoRT run.

It uses recreated artifacts only: a T5 AST/prefix compiler trained from public AST demonstrations and a weak post-judgment classifier trained from public WMDP-derived proxy labels. It is not an official paper checkpoint reproduction and must not be used as the final paper metric.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'joblib': 'joblib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'safetensors': 'safetensors',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'transformers': 'transformers>=4.38.0',
    'sentencepiece': 'sentencepiece',
    'yaml': 'pyyaml',
    'tqdm': 'tqdm',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


## Runtime Config

Default behavior runs on a clean Kaggle GPU session. The runner first tries to resolve recreated artifacts from `PORT_RECREATED_ARTIFACT_DIR`, `PORT_RECREATED_ARTIFACT_ZIP_URL`, or `PORT_RECREATED_ARTIFACT_ZIP_PATH`; if none is available, it bootstraps the recreated artifacts in this notebook from public repo data.

Useful overrides:

- `PORT_WMDP_VARIANTS=original,noise_prefix,composite`
- `PORT_WMDP_DOMAINS=bio,chem,cyber`
- `PORT_MAX_SAMPLES=2`
- `PORT_RECREATED_ARTIFACT_ZIP_URL=https://.../paper_port_recreated_artifacts_bootstrap.zip`
- `PORT_RECREATED_ARTIFACT_DIR=/kaggle/working/paper_port_recreated_artifacts_bootstrap`


In [ ]:
import importlib.util

runner_path = PROJECT_ROOT / 'notebooks' / 'common' / 'port_recreated_smoke.py'
if not runner_path.exists():
    raise FileNotFoundError(runner_path)

spec = importlib.util.spec_from_file_location('port_recreated_smoke', runner_path)
port_recreated_smoke = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_recreated_smoke)

result = port_recreated_smoke.run(
    project_root=PROJECT_ROOT,
    is_kaggle=IS_KAGGLE,
    commit_sha=commit_sha,
)
print(json.dumps(result, indent=2, default=str))
